In [1]:
#Project 2 Lede Program
#Marcela Vilar Mota Santos

In [1]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
import geocoder
from datetime import datetime, timedelta, timezone
import json
import time
from zoneinfo import ZoneInfo

## getting the data from the world cup matches and basic info about it

In [2]:
url = "https://www.zoho.com/toolkit/fifa-world-cup-2026.html?zredirect=f&zsrc=langdropdown&lb=pt-br&lb=pt-br"

In [3]:
table = pd.read_html(url)

In [4]:
stadiums = table[0]
stadiums.head(3)

,Host Country,Host City,Official FIFA Stadium Designation,Net Capacity
0,United States,New York/New Jersey,New York New Jersey Stadium,78576
1,United States,Kansas City,Kansas City Stadium,67513
2,United States,San Francisco Bay Area,San Francisco Bay Area Stadium,69391


In [5]:
stadiums.shape

(16, 4)

In [6]:
stadiums.sort_values(by="Net Capacity", ascending=False)
#New York New Jersey Stadium biggest stadium

,Host Country,Host City,Official FIFA Stadium Designation,Net Capacity
0,United States,New York/New Jersey,New York New Jersey Stadium,78576
13,Mexico,Mexico City,Mexico City Stadium,72766
6,United States,Dallas,Dallas Stadium,70122
7,United States,Los Angeles,Los Angeles Stadium,69650
2,United States,San Francisco Bay Area,San Francisco Bay Area Stadium,69391
5,United States,Houston,Houston Stadium,68311
1,United States,Kansas City,Kansas City Stadium,67513
3,United States,Atlanta,Atlanta Stadium,67382
8,United States,Philadelphia,Philadelphia Stadium,65827
10,United States,Seattle,Seattle Stadium,65123


In [7]:
prize = table[4]
prize.head(3)

,Final Position,Prize Money (per Team)
0,Champions (Winner),$50 million
1,Runner-up,$33 million
2,Third place,$29 million


In [8]:
groups = table[5]
groups.head(3)

,Group,Participating Nations
0,Group A,"Mexico, South Africa, South Korea, Czechia"
1,Group B,"Canada, Bosnia and Herzegovina, Switzerland, Q..."
2,Group C,"Brazil, Morocco, Scotland, Haiti"


In [10]:
groups['Participating Nations'] = groups['Participating Nations'].str.split(',')

In [11]:
groups

,Group,Participating Nations
0,Group A,"[Mexico, South Africa, South Korea, Czechia]"
1,Group B,"[Canada, Bosnia and Herzegovina, Switzerland..."
2,Group C,"[Brazil, Morocco, Scotland, Haiti]"
3,Group D,"[USA, Paraguay, Australia, Türkiye]"
4,Group E,"[Germany, Ecuador, Ivory Coast, Curaçao]"
5,Group F,"[Netherlands, Japan, Tunisia, Sweden]"
6,Group G,"[Belgium, Iran, Egypt, New Zealand]"
7,Group H,"[Spain, Cabo Verde, Saudi Arabia, Uruguay]"
8,Group I,"[France, Senegal, Iraq, Norway]"
9,Group J,"[Argentina, Algeria, Austria, Jordan]"


In [13]:
groups = groups.explode('Participating Nations')       
groups['Participating Nations'] = groups['Participating Nations'].str.strip()     
groups = groups.reset_index(drop=True)

In [14]:
groups

,Group,Participating Nations
0,Group A,Mexico
1,Group A,South Africa
2,Group A,South Korea
3,Group A,Czechia
4,Group B,Canada
5,Group B,Bosnia and Herzegovina
6,Group B,Switzerland
7,Group B,Qatar
8,Group C,Brazil
9,Group C,Morocco


In [10]:
matches = table[6][1:]
matches.head(3)

,Match,Date,Day,Group Stage,Team 1,vs,Team 2,Venue,City,Region
1,1,Jun 11,Thu,A,Mexico,vs,South Africa,Mexico City Stadium,Mexico City,Central
2,2,Jun 11,Thu,A,South Korea,vs,Czech Republic,Estadio Guadalajara,Guadalajara,Central
3,3,Jun 12,Fri,B,Canada,vs,Bosnia and Herzegovina,Toronto Stadium,Toronto,Eastern


In [11]:
matches.tail(5)
#the final will be in the new jersey stadium

,Match,Date,Day,Group Stage,Team 1,vs,Team 2,Venue,City,Region
106,102,Jul 15,Wed,Semi-final,Winner Match 99,vs,Winner Match 100,Atlanta Stadium,Atlanta,Eastern
107,"THIRD-PLACE MATCH (July 18, 2026)","THIRD-PLACE MATCH (July 18, 2026)","THIRD-PLACE MATCH (July 18, 2026)","THIRD-PLACE MATCH (July 18, 2026)","THIRD-PLACE MATCH (July 18, 2026)","THIRD-PLACE MATCH (July 18, 2026)","THIRD-PLACE MATCH (July 18, 2026)","THIRD-PLACE MATCH (July 18, 2026)","THIRD-PLACE MATCH (July 18, 2026)","THIRD-PLACE MATCH (July 18, 2026)"
108,103,Jul 18,Sat,Third Place,Loser Match 101,vs,Loser Match 102,Miami Stadium,Miami,Eastern
109,"FINAL (July 19, 2026)","FINAL (July 19, 2026)","FINAL (July 19, 2026)","FINAL (July 19, 2026)","FINAL (July 19, 2026)","FINAL (July 19, 2026)","FINAL (July 19, 2026)","FINAL (July 19, 2026)","FINAL (July 19, 2026)","FINAL (July 19, 2026)"
110,104,Jul 19,Sun,Final,Winner Match 101,vs,Winner Match 102,New York New Jersey Stadium,New York/NJ,Eastern


In [12]:
#droping columns
matches = matches.drop(columns =['Day','Group Stage', 'vs', "Region"])

In [13]:
matches.head(3)

,Match,Date,Team 1,Team 2,Venue,City
1,1,Jun 11,Mexico,South Africa,Mexico City Stadium,Mexico City
2,2,Jun 11,South Korea,Czech Republic,Estadio Guadalajara,Guadalajara
3,3,Jun 12,Canada,Bosnia and Herzegovina,Toronto Stadium,Toronto


In [14]:
len(matches)

110

In [15]:
#finding irregular columns
matches["Match"][109]

'FINAL (July 19, 2026)'

In [16]:
matches["Match"][107]

'THIRD-PLACE MATCH (July 18, 2026)'

In [17]:
matches["Match"][104]

'SEMI-FINALS (July 14 – July 15, 2026)'

In [18]:
matches["Match"][99]

'QUARTER-FINALS (July 9 – July 11, 2026)'

In [19]:
matches["Match"][90]

'ROUND OF 16 (July 4 – July 7, 2026)'

In [20]:
matches["Match"][73]

'ROUND OF 32 (June 28 – July 3, 2026)'

In [21]:
#converting to datetime format
#adding the year and converting to datetime
matches["Date"] = pd.to_datetime(matches["Date"], format="%b %d", errors="coerce")

#formating as MM-DD-YYYY
matches["Date"] = matches["Date"].dt.strftime("%m-%d-2026")

In [22]:
matches.head(3)

,Match,Date,Team 1,Team 2,Venue,City
1,1,06-11-2026,Mexico,South Africa,Mexico City Stadium,Mexico City
2,2,06-11-2026,South Korea,Czech Republic,Estadio Guadalajara,Guadalajara
3,3,06-12-2026,Canada,Bosnia and Herzegovina,Toronto Stadium,Toronto


## separating by phase 

In [23]:
#creating a new empty column
matches["phase"] = None

In [24]:
matches.head(3)

,Match,Date,Team 1,Team 2,Venue,City,phase
1,1,06-11-2026,Mexico,South Africa,Mexico City Stadium,Mexico City,None
2,2,06-11-2026,South Korea,Czech Republic,Estadio Guadalajara,Guadalajara,None
3,3,06-12-2026,Canada,Bosnia and Herzegovina,Toronto Stadium,Toronto,None


In [25]:
matches["Date"].dtype

<StringDtype(na_value=nan)>

In [26]:
#transforming to datetime element so the for loop works
matches["Date"] = pd.to_datetime(matches['Date'])

In [27]:
matches["Date"].dtype

dtype('<M8[us]')

In [28]:
#filling the new column
for index, row in matches.iterrows(): 
    if pd.Timestamp("2026-06-11") <= row["Date"] <= pd.Timestamp("2026-06-27"):
       matches.at[index, "phase"] = "Group Stage"
    elif pd.Timestamp("2026-06-28") <= row["Date"] <= pd.Timestamp("2026-07-03"):
       matches.at[index, "phase"] = "Round of 32"
    elif pd.Timestamp("2026-07-04") <= row["Date"] <= pd.Timestamp("2026-07-07"):
       matches.at[index, "phase"] = "Round of 16"
    elif pd.Timestamp("2026-07-09") <= row["Date"] <= pd.Timestamp("2026-07-11"):
       matches.at[index, "phase"] = "Quarterfinals"
    elif pd.Timestamp("2026-07-14") <= row["Date"] <= pd.Timestamp("2026-07-15"):
       matches.at[index, "phase"] = "Semifinals"
    elif pd.Timestamp("2026-07-18") == row["Date"]:
       matches.at[index, "phase"] = "Third Place"
    elif pd.Timestamp("2026-07-19") == row["Date"]:
       matches.at[index, "phase"] = "Final"

In [29]:
matches["phase"].unique()

array(['Group Stage', None, 'Round of 32', 'Round of 16', 'Quarterfinals',
       'Semifinals', 'Third Place', 'Final'], dtype=object)

In [30]:
matches[matches["phase"].isna()]

,Match,Date,Team 1,Team 2,Venue,City,phase
73,"ROUND OF 32 (June 28 – July 3, 2026)",NaT,"ROUND OF 32 (June 28 – July 3, 2026)","ROUND OF 32 (June 28 – July 3, 2026)","ROUND OF 32 (June 28 – July 3, 2026)","ROUND OF 32 (June 28 – July 3, 2026)",None
90,"ROUND OF 16 (July 4 – July 7, 2026)",NaT,"ROUND OF 16 (July 4 – July 7, 2026)","ROUND OF 16 (July 4 – July 7, 2026)","ROUND OF 16 (July 4 – July 7, 2026)","ROUND OF 16 (July 4 – July 7, 2026)",None
99,"QUARTER-FINALS (July 9 – July 11, 2026)",NaT,"QUARTER-FINALS (July 9 – July 11, 2026)","QUARTER-FINALS (July 9 – July 11, 2026)","QUARTER-FINALS (July 9 – July 11, 2026)","QUARTER-FINALS (July 9 – July 11, 2026)",None
104,"SEMI-FINALS (July 14 – July 15, 2026)",NaT,"SEMI-FINALS (July 14 – July 15, 2026)","SEMI-FINALS (July 14 – July 15, 2026)","SEMI-FINALS (July 14 – July 15, 2026)","SEMI-FINALS (July 14 – July 15, 2026)",None
107,"THIRD-PLACE MATCH (July 18, 2026)",NaT,"THIRD-PLACE MATCH (July 18, 2026)","THIRD-PLACE MATCH (July 18, 2026)","THIRD-PLACE MATCH (July 18, 2026)","THIRD-PLACE MATCH (July 18, 2026)",None
109,"FINAL (July 19, 2026)",NaT,"FINAL (July 19, 2026)","FINAL (July 19, 2026)","FINAL (July 19, 2026)","FINAL (July 19, 2026)",None


In [31]:
matches["phase"].value_counts() #checking if it worked

phase
Group Stage      72
Round of 32      16
Round of 16       8
Quarterfinals     4
Semifinals        2
Third Place       1
Final             1
Name: count, dtype: int64

In [32]:
matches.to_csv("matches.csv", index=False)

## hiding my geocoder API key 

In [33]:
#hiding my api key
# create a file called `.env` (# it's a hidden file)
# you also will want to create a file called .gitignore (it is also hidden)
# and put this in it (without the #)
# .env 
# that's to tell Git not to put your .env file (with your API key) onto Github

In [16]:
load_dotenv()

True

In [35]:
ls

 O volume na unidade C ‚ OS
 O N£mero de S‚rie do Volume ‚ 44F5-4363

 Pasta de C:\Users\samue\OneDrive\Desktop\Lede\Projects\world cup

07/09/2026  07:01 PM    <DIR>          .
07/06/2026  02:54 PM    <DIR>          ..
07/09/2026  08:46 AM               191 .env
07/05/2026  11:58 PM             4,706 .gitignore
07/09/2026  03:12 PM    <DIR>          .ipynb_checkpoints
07/09/2026  10:34 AM         4,169,740 2026.csv
07/05/2026  11:49 PM                60 api key geocoder.txt
07/09/2026  10:55 AM                85 credentials.json
07/09/2026  04:40 PM             1,122 df_geo.csv
07/09/2026  09:32 AM            48,234 DFW.pdf
07/03/2026  05:04 PM             4,727 extract_worldcup.py
07/03/2026  03:59 PM            44,443 fifa-world-cup-2026.ics
07/09/2026  05:17 PM         5,633,771 flights_LGA.json
07/09/2026  09:47 AM           423,907 ITA.xlsx
07/09/2026  10:01 AM           423,907 ITA_monthly.xlsx
07/09/2026  07:01 PM             9,399 matches.csv
07/03/2026  06:23 PM              

In [20]:
api_key_geocoder = os.environ["api_key_geocoder"]

In [37]:
api_key_geocoder[:9]+"..."

'AIzaSyDI7...'

In [39]:
#making a new df with only the venue and city, because it may be easier for the geocoder to read 
df_geo = stadiums[['Official FIFA Stadium Designation', 'Host City', 'Host Country']]
df_geo.head(3)

,Official FIFA Stadium Designation,Host City,Host Country
0,New York New Jersey Stadium,New York/New Jersey,United States
1,Kansas City Stadium,Kansas City,United States
2,San Francisco Bay Area Stadium,San Francisco Bay Area,United States


In [40]:
#changing the name of the column 
df_geo.rename(columns={"Host City":"City", 
                       "Official FIFA Stadium Designation": "Venue", 
                        "Host Country" : "Country"}, inplace=True)

In [41]:
df_geo.head(3)
#checking if it worked

,Venue,City,Country
0,New York New Jersey Stadium,New York/New Jersey,United States
1,Kansas City Stadium,Kansas City,United States
2,San Francisco Bay Area Stadium,San Francisco Bay Area,United States


In [42]:
df_geo.to_csv("df_geo.csv", index=False)

## geocoding

In [43]:
#getting the latitude and longitude
for index, row in df_geo.iterrows():
    full_address = f"{row['Venue']}, {row['City']}, {row['Country']}"
    print(full_address)
    
    geo = geocoder.google(full_address, key=api_key_geocoder)
    print("Status:", geo.status)
    print("OK:", geo.ok)
    print("LatLng:", geo.latlng)
    
    if geo.ok:
        df_geo.loc[index, "lat"] = geo.latlng[0]
        df_geo.loc[index, "long"] = geo.latlng[1]
    else:
        print(f"Failed for {full_address}: {geo.status}")

df_geo

New York New Jersey Stadium, New York/New Jersey, United States
Status: OK
OK: True
LatLng: [40.8135064, -74.074457]
Kansas City Stadium, Kansas City, United States
Status: OK
OK: True
LatLng: [39.0489647, -94.4840141]
San Francisco Bay Area Stadium, San Francisco Bay Area, United States
Status: OK
OK: True
LatLng: [37.4033165, -121.9693774]
Atlanta Stadium, Atlanta, United States
Status: OK
OK: True
LatLng: [33.7553232, -84.40059049999999]
Miami Stadium, Miami, United States
Status: OK
OK: True
LatLng: [25.7616798, -80.1917902]
Houston Stadium, Houston, United States
Status: OK
OK: True
LatLng: [29.6847219, -95.41070739999999]
Dallas Stadium, Dallas, United States
Status: OK
OK: True
LatLng: [32.7480281, -97.09281240000001]
Los Angeles Stadium, Los Angeles, United States
Status: OK
OK: True
LatLng: [33.9534765, -118.3390235]
Philadelphia Stadium, Philadelphia, United States
Status: OK
OK: True
LatLng: [39.9014189, -75.1675992]
Boston Stadium, Boston, United States
Status: OK
OK: True


,Venue,City,Country,lat,long
0,New York New Jersey Stadium,New York/New Jersey,United States,40.813506,-74.074457
1,Kansas City Stadium,Kansas City,United States,39.048965,-94.484014
2,San Francisco Bay Area Stadium,San Francisco Bay Area,United States,37.403317,-121.969377
3,Atlanta Stadium,Atlanta,United States,33.755323,-84.400590
4,Miami Stadium,Miami,United States,25.761680,-80.191790
5,Houston Stadium,Houston,United States,29.684722,-95.410707
6,Dallas Stadium,Dallas,United States,32.748028,-97.092812
7,Los Angeles Stadium,Los Angeles,United States,33.953477,-118.339023
8,Philadelphia Stadium,Philadelphia,United States,39.901419,-75.167599
9,Boston Stadium,Boston,United States,42.090927,-71.264363


In [44]:
df_geo.to_csv("df_geo.csv", index=False)

In [18]:
groups

,Group,Participating Nations
0,Group A,Mexico
1,Group A,South Africa
2,Group A,South Korea
3,Group A,Czechia
4,Group B,Canada
5,Group B,Bosnia and Herzegovina
6,Group B,Switzerland
7,Group B,Qatar
8,Group C,Brazil
9,Group C,Morocco


In [22]:
#geocoding the participating nations

for index, row in groups.iterrows():
    participating_nation = f"{row['Participating Nations']}"
    print(participating_nation)
    
    geo = geocoder.google(participating_nation, key=api_key_geocoder)
    print("Status:", geo.status)
    print("OK:", geo.ok)
    print("LatLng:", geo.latlng)
    
    if geo.ok:
        groups.loc[index, "lat"] = geo.latlng[0]
        groups.loc[index, "long"] = geo.latlng[1]
    else:
        print(f"Failed for {participating_nation}: {geo.status}")

groups

Mexico
Status: OK
OK: True
LatLng: [23.634501, -102.552784]
South Africa
Status: OK
OK: True
LatLng: [-30.559482, 22.937506]
South Korea
Status: OK
OK: True
LatLng: [35.907757, 127.766922]
Czechia
Status: OK
OK: True
LatLng: [49.81749199999999, 15.472962]
Canada
Status: OK
OK: True
LatLng: [56.130366, -106.346771]
Bosnia and Herzegovina
Status: OK
OK: True
LatLng: [43.915886, 17.679076]
Switzerland
Status: OK
OK: True
LatLng: [46.818188, 8.227511999999999]
Qatar
Status: OK
OK: True
LatLng: [25.354826, 51.183884]
Brazil
Status: OK
OK: True
LatLng: [-14.235004, -51.92528]
Morocco
Status: OK
OK: True
LatLng: [31.791702, -7.092619999999999]
Scotland
Status: OK
OK: True
LatLng: [57.7470127655324, -4.687305781018138]
Haiti
Status: OK
OK: True
LatLng: [18.971187, -72.285215]
USA
Status: OK
OK: True
LatLng: [38.7945952, -106.5348379]
Paraguay
Status: OK
OK: True
LatLng: [-23.442503, -58.443832]
Australia
Status: OK
OK: True
LatLng: [-25.274398, 133.775136]
Türkiye
Status: OK
OK: True
LatLng: [

,Group,Participating Nations,lat,long
0,Group A,Mexico,23.634501,-102.552784
1,Group A,South Africa,-30.559482,22.937506
2,Group A,South Korea,35.907757,127.766922
3,Group A,Czechia,49.817492,15.472962
4,Group B,Canada,56.130366,-106.346771
5,Group B,Bosnia and Herzegovina,43.915886,17.679076
6,Group B,Switzerland,46.818188,8.227512
7,Group B,Qatar,25.354826,51.183884
8,Group C,Brazil,-14.235004,-51.925280
9,Group C,Morocco,31.791702,-7.092620


In [24]:
groups.to_csv("participating_nations.csv", index=False)

## doing some analysis

In [45]:
matches.head()

,Match,Date,Team 1,Team 2,Venue,City,phase
1,1,2026-06-11,Mexico,South Africa,Mexico City Stadium,Mexico City,Group Stage
2,2,2026-06-11,South Korea,Czech Republic,Estadio Guadalajara,Guadalajara,Group Stage
3,3,2026-06-12,Canada,Bosnia and Herzegovina,Toronto Stadium,Toronto,Group Stage
4,4,2026-06-12,USA,Paraguay,Los Angeles Stadium,Los Angeles,Group Stage
5,5,2026-06-13,Qatar,Switzerland,San Francisco Bay Area Stadium,San Francisco Bay Area,Group Stage


In [46]:
matches_by_stadium = matches["Venue"].value_counts()
matches_by_stadium
#Dallas will host more matches

Venue
Dallas Stadium                             9
Los Angeles Stadium                        8
New York New Jersey Stadium                8
Atlanta Stadium                            8
Miami Stadium                              8
Boston Stadium                             7
BC Place Vancouver                         7
Mexico City Stadium                        6
San Francisco Bay Area Stadium             6
Houston Stadium                            6
Philadelphia Stadium                       6
Seattle Stadium                            6
Kansas City Stadium                        6
Toronto Stadium                            5
Estadio Guadalajara                        4
Estadio Monterrey                          4
ROUND OF 32 (June 28 – July 3, 2026)       1
ROUND OF 16 (July 4 – July 7, 2026)        1
QUARTER-FINALS (July 9 – July 11, 2026)    1
SEMI-FINALS (July 14 – July 15, 2026)      1
THIRD-PLACE MATCH (July 18, 2026)          1
FINAL (July 19, 2026)                      1
Name

In [106]:
matches_by_stadium.to_csv("matches_by_stadium.csv")

In [47]:
matches_per_day = matches.groupby("Date")["Match"].count().sort_values(ascending=False)
matches_per_day 
#number of games per day during the whole world cup

Date
2026-06-26    6
2026-06-25    6
2026-06-27    6
2026-06-24    6
2026-06-20    4
2026-06-19    4
2026-06-13    4
2026-06-14    4
2026-06-15    4
2026-06-16    4
2026-06-23    4
2026-06-22    4
2026-06-18    4
2026-06-17    4
2026-06-21    4
2026-06-30    3
2026-07-03    3
2026-06-29    3
2026-07-01    3
2026-07-02    3
2026-06-12    2
2026-06-11    2
2026-07-07    2
2026-07-04    2
2026-07-05    2
2026-07-06    2
2026-07-11    2
2026-06-28    1
2026-07-09    1
2026-07-10    1
2026-07-14    1
2026-07-15    1
2026-07-18    1
2026-07-19    1
Name: Match, dtype: int64

In [48]:
matches_NY = matches[matches["City"] == "New York/NJ"]
matches_NY

,Match,Date,Team 1,Team 2,Venue,City,phase
6,6,2026-06-13,Brazil,Morocco,New York New Jersey Stadium,New York/NJ,Group Stage
17,17,2026-06-16,France,Senegal,New York New Jersey Stadium,New York/NJ,Group Stage
41,41,2026-06-22,Norway,Senegal,New York New Jersey Stadium,New York/NJ,Group Stage
56,56,2026-06-25,Ecuador,Germany,New York New Jersey Stadium,New York/NJ,Group Stage
67,67,2026-06-27,Panama,England,New York New Jersey Stadium,New York/NJ,Group Stage
78,77,2026-06-30,Group I Winner,3rd Place (C/D/F/G/H),New York New Jersey Stadium,New York/NJ,Round of 32
93,91,2026-07-05,Winner Match 76,Winner Match 78,New York New Jersey Stadium,New York/NJ,Round of 16
110,104,2026-07-19,Winner Match 101,Winner Match 102,New York New Jersey Stadium,New York/NJ,Final


In [49]:
matches_NY = matches_NY.drop(columns = ["Match", "City"])
matches_NY

,Date,Team 1,Team 2,Venue,phase
6,2026-06-13,Brazil,Morocco,New York New Jersey Stadium,Group Stage
17,2026-06-16,France,Senegal,New York New Jersey Stadium,Group Stage
41,2026-06-22,Norway,Senegal,New York New Jersey Stadium,Group Stage
56,2026-06-25,Ecuador,Germany,New York New Jersey Stadium,Group Stage
67,2026-06-27,Panama,England,New York New Jersey Stadium,Group Stage
78,2026-06-30,Group I Winner,3rd Place (C/D/F/G/H),New York New Jersey Stadium,Round of 32
93,2026-07-05,Winner Match 76,Winner Match 78,New York New Jersey Stadium,Round of 16
110,2026-07-19,Winner Match 101,Winner Match 102,New York New Jersey Stadium,Final


In [84]:
matches_NY.to_csv("matches_NY.csv", index=False)

## getting data from the air traffic in NYC (OPenSky API)

https://openskynetwork.github.io/opensky-api/rest.html 

In [50]:
airports_ny = {"airport_1" : "John F. Kennedy International Airport (JFK)",
                "ICAO_code_1" : "KJFK",
                "airport_2" : "LaGuardia Airport (LGA)",
                "ICAO_code_2" : "KLGA", 
                "airport_3" : "Newark International Airport (EWR)",
                "ICAO_code_3" : "KEWR", }

In [15]:
#I wanna analyse how is the air traffic one week before the first game in NY
#Until the day before the last game
#from june 6 to july 6
#begin and end are not dates or ISO timestamps
#must be Unix timestamps (seconds since January 1, 1970 UTC)
#API limitation = each request must be made within a maximum interval of 48 hours

start = datetime(2026, 6, 6)
end = datetime(2026, 7, 6)

intervals = []

current = start

while current < end:
    next_date = min(current + timedelta(days=1), end)

    intervals.append((current, next_date))

    current = next_date

print(intervals)

[(datetime.datetime(2026, 6, 6, 0, 0), datetime.datetime(2026, 6, 7, 0, 0)), (datetime.datetime(2026, 6, 7, 0, 0), datetime.datetime(2026, 6, 8, 0, 0)), (datetime.datetime(2026, 6, 8, 0, 0), datetime.datetime(2026, 6, 9, 0, 0)), (datetime.datetime(2026, 6, 9, 0, 0), datetime.datetime(2026, 6, 10, 0, 0)), (datetime.datetime(2026, 6, 10, 0, 0), datetime.datetime(2026, 6, 11, 0, 0)), (datetime.datetime(2026, 6, 11, 0, 0), datetime.datetime(2026, 6, 12, 0, 0)), (datetime.datetime(2026, 6, 12, 0, 0), datetime.datetime(2026, 6, 13, 0, 0)), (datetime.datetime(2026, 6, 13, 0, 0), datetime.datetime(2026, 6, 14, 0, 0)), (datetime.datetime(2026, 6, 14, 0, 0), datetime.datetime(2026, 6, 15, 0, 0)), (datetime.datetime(2026, 6, 15, 0, 0), datetime.datetime(2026, 6, 16, 0, 0)), (datetime.datetime(2026, 6, 16, 0, 0), datetime.datetime(2026, 6, 17, 0, 0)), (datetime.datetime(2026, 6, 17, 0, 0), datetime.datetime(2026, 6, 18, 0, 0)), (datetime.datetime(2026, 6, 18, 0, 0), datetime.datetime(2026, 6, 19, 

In [16]:
def to_unix(day):
    return int(day.replace(tzinfo=ZoneInfo("America/New_York")).timestamp())

unix_intervals = [(to_unix(inicio), to_unix(fim)) for inicio, fim in intervals]

for begin, end in unix_intervals:
    print(begin, end)

1780718400 1780804800
1780804800 1780891200
1780891200 1780977600
1780977600 1781064000
1781064000 1781150400
1781150400 1781236800
1781236800 1781323200
1781323200 1781409600
1781409600 1781496000
1781496000 1781582400
1781582400 1781668800
1781668800 1781755200
1781755200 1781841600
1781841600 1781928000
1781928000 1782014400
1782014400 1782100800
1782100800 1782187200
1782187200 1782273600
1782273600 1782360000
1782360000 1782446400
1782446400 1782532800
1782532800 1782619200
1782619200 1782705600
1782705600 1782792000
1782792000 1782878400
1782878400 1782964800
1782964800 1783051200
1783051200 1783137600
1783137600 1783224000
1783224000 1783310400


In [17]:
len(unix_intervals)

30

In [3]:
#activing credentials for the API key
client_id = os.environ["client_id"]

In [5]:
client_id[:3]+"..."

'cel...'

In [6]:
client_secret = os.environ["client_secret"]

In [7]:
client_secret[:3]+"..."

'1NA...'

In [8]:
client_id_2 = os.environ["client_id_2"]

In [9]:
client_id_2[:3]+"..."

'mvi...'

In [10]:
client_secret_2 = os.environ["client_secret_2"]

In [11]:
client_secret_2[:3]+"..."

'0p6...'

In [12]:
CLIENT_ID = client_id
CLIENT_SECRET = client_secret

TOKEN_URL = "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token"

def get_token():
    r = requests.post(
        TOKEN_URL,
        data={
            "grant_type": "client_credentials",
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
        },
    )
    r.raise_for_status()
    return r.json()["access_token"]

token = get_token()

In [13]:
API_URL = "https://opensky-network.org/api/flights/arrival"
headers = {"Authorization": f"Bearer {token}"}

def get_flights_for_airport(airport):
    
    flights_airport = []
    
    for begin, end in unix_intervals:
        resp = requests.get(
            API_URL,
            params={"airport": airport, "begin": begin, "end": end},
            headers=headers)

        if resp.status_code == 404:
            print(f"No flights were found for {airport} ({begin}-{end})")
            continue

        resp.raise_for_status()
        flights = resp.json()
        print(f"{airport} {begin}-{end}: {len(flights)} flights found")
        flights_airport.extend(flights)
        time.sleep(1)

    return flights_airport

## getting flights from la guardia

In [18]:
flights_la_guardia = get_flights_for_airport("KLGA")
with open("flights_LGA.json", "w") as f:
    json.dump(flights_la_guardia, f)

KLGA 1780718400-1780804800: 322 flights found
KLGA 1780804800-1780891200: 477 flights found
KLGA 1780891200-1780977600: 583 flights found
KLGA 1780977600-1781064000: 550 flights found
KLGA 1781064000-1781150400: 464 flights found
KLGA 1781150400-1781236800: 465 flights found
KLGA 1781236800-1781323200: 394 flights found
KLGA 1781323200-1781409600: 387 flights found
KLGA 1781409600-1781496000: 394 flights found
KLGA 1781496000-1781582400: 539 flights found
KLGA 1781582400-1781668800: 544 flights found
KLGA 1781668800-1781755200: 545 flights found
KLGA 1781755200-1781841600: 564 flights found
KLGA 1781841600-1781928000: 564 flights found
KLGA 1781928000-1782014400: 361 flights found
KLGA 1782014400-1782100800: 532 flights found
KLGA 1782100800-1782187200: 385 flights found
KLGA 1782187200-1782273600: 540 flights found
KLGA 1782273600-1782360000: 577 flights found
KLGA 1782360000-1782446400: 580 flights found
KLGA 1782446400-1782532800: 577 flights found
KLGA 1782532800-1782619200: 363 fl

In [58]:
flights_la_guardia[0]

{'icao24': 'abe79d',
 'firstSeen': 1780793531,
 'estDepartureAirport': 'KDEN',
 'lastSeen': 1780804798,
 'estArrivalAirport': 'KLGA',
 'callsign': 'SWA4480 ',
 'estDepartureAirportHorizDistance': 4827,
 'estDepartureAirportVertDistance': 89,
 'estArrivalAirportHorizDistance': 377,
 'estArrivalAirportVertDistance': 24,
 'departureAirportCandidatesCount': 5,
 'arrivalAirportCandidatesCount': 3}

In [59]:
df_laguardia = pd.DataFrame(flights_la_guardia)
df_laguardia

,icao24,firstSeen,estDepartureAirport,lastSeen,estArrivalAirport,callsign,estDepartureAirportHorizDistance,estDepartureAirportVertDistance,estArrivalAirportHorizDistance,estArrivalAirportVertDistance,departureAirportCandidatesCount,arrivalAirportCandidatesCount
0,abe79d,1780793531,KDEN,1780804798,KLGA,SWA4480,4827.0,89.0,377,24,5,3
1,abd230,1780797756,KBNA,1780804585,KLGA,SWA2144,3146.0,183.0,381,16,0,3
2,a0540a,1780802023,KBWI,1780804357,KLGA,DAL997,831.0,24.0,393,31,1,3
3,abc4be,1780793350,KDAL,1780803735,KLGA,SWA3256,2003.0,247.0,418,16,1,3
4,ac6227,1780796317,KSTL,1780803608,KLGA,SWA4684,1441.0,5.0,440,24,1,3
...,...,...,...,...,...,...,...,...,...,...,...,...
14577,ac81cc,1783222721,KORD,1783228818,KLGA,AAL2086,1699.0,6.0,1443,1,0,3
14578,a22358,1783222641,KCMH,1783227896,KLGA,RPA5784,1321.0,33.0,247,6,0,3
14579,ad8935,1783213820,KDFW,1783225456,KLGA,AAL2708,1075.0,66.0,523,29,3,3
14580,ac23ef,1783223125,KALB,1783224929,KLGA,SWA2144,1478.0,111.0,1716,1,2,3


In [60]:
df_laguardia = df_laguardia.drop(columns = ["icao24", "estDepartureAirportHorizDistance", "estDepartureAirportVertDistance", "estArrivalAirportHorizDistance", "estArrivalAirportVertDistance", "departureAirportCandidatesCount", "arrivalAirportCandidatesCount"])

In [61]:
df_laguardia = df_laguardia.rename(columns = {"firstSeen" : "departure_time",
                               "estDepartureAirport" : "departure_airport", 
                               "lastSeen" : "arrival_time",
                               "estArrivalAirport" : "arrival_airport"})
df_laguardia

,departure_time,departure_airport,arrival_time,arrival_airport,callsign
0,1780793531,KDEN,1780804798,KLGA,SWA4480
1,1780797756,KBNA,1780804585,KLGA,SWA2144
2,1780802023,KBWI,1780804357,KLGA,DAL997
3,1780793350,KDAL,1780803735,KLGA,SWA3256
4,1780796317,KSTL,1780803608,KLGA,SWA4684
...,...,...,...,...,...
14577,1783222721,KORD,1783228818,KLGA,AAL2086
14578,1783222641,KCMH,1783227896,KLGA,RPA5784
14579,1783213820,KDFW,1783225456,KLGA,AAL2708
14580,1783223125,KALB,1783224929,KLGA,SWA2144


In [63]:
#converting arrival time
df_laguardia["arrival_time_converted"] = (
    pd.to_datetime(df_laguardia["arrival_time"], unit="s", utc=True) #unit = s says that is in seconds
    .dt.tz_convert("America/New_York") #converts to ny timezone                       
    .dt.strftime("%Y-%m-%d %H:%M")
)

In [ ]:
#converting departure time
df_laguardia["departure_time_converted"] = (
    pd.to_datetime(df_laguardia["departure_time"], unit="s", utc=True) #unit = s says that is in seconds
    .dt.tz_convert("America/New_York") #converts to ny timezone                       
    .dt.strftime("%Y-%m-%d %H:%M")
)

In [64]:
df_laguardia["arrival_time_converted"].sort_values()

321      2026-06-06 06:51
320      2026-06-06 07:01
319      2026-06-06 07:05
318      2026-06-06 07:07
317      2026-06-06 07:09
               ...       
14181    2026-07-05 22:43
14180    2026-07-05 22:50
14179    2026-07-05 22:57
14178    2026-07-05 23:11
14177    2026-07-05 23:15
Name: arrival_time_converted, Length: 14582, dtype: str

In [65]:
df_laguardia[["data", "horario"]] = df_laguardia["arrival_time_converted"].str.split(" ", expand=True)

In [66]:
df_laguardia.head()

,departure_time,departure_airport,arrival_time,arrival_airport,callsign,departure_time_converted,arrival_time_converted,data,horario
0,1780793531,KDEN,1780804798,KLGA,SWA4480,2026-06-06 20:52,2026-06-06 23:59,2026-06-06,23:59
1,1780797756,KBNA,1780804585,KLGA,SWA2144,2026-06-06 22:02,2026-06-06 23:56,2026-06-06,23:56
2,1780802023,KBWI,1780804357,KLGA,DAL997,2026-06-06 23:13,2026-06-06 23:52,2026-06-06,23:52
3,1780793350,KDAL,1780803735,KLGA,SWA3256,2026-06-06 20:49,2026-06-06 23:42,2026-06-06,23:42
4,1780796317,KSTL,1780803608,KLGA,SWA4684,2026-06-06 21:38,2026-06-06 23:40,2026-06-06,23:40


In [67]:
df_laguardia = df_laguardia.rename(columns={"departure_time": "dep_unix",
                            "arrival_time" : "arrival_unix",
                             "departure_time_converted" : "dep_converted",
                             "arrival_time_converted" : "arrival_conv",
                             "data": "arrival",
                             "horario" : "ar_time"})

In [68]:
df_laguardia =  df_laguardia.rename(columns={"departure_airport" : "dep_airport",
                                            "arrival_airport" : "arr_airport",
                                            "dep_converted" : "dep_conv"})

In [69]:
df_laguardia

,dep_unix,dep_airport,arrival_unix,arr_airport,callsign,dep_conv,arrival_conv,arrival,ar_time
0,1780793531,KDEN,1780804798,KLGA,SWA4480,2026-06-06 20:52,2026-06-06 23:59,2026-06-06,23:59
1,1780797756,KBNA,1780804585,KLGA,SWA2144,2026-06-06 22:02,2026-06-06 23:56,2026-06-06,23:56
2,1780802023,KBWI,1780804357,KLGA,DAL997,2026-06-06 23:13,2026-06-06 23:52,2026-06-06,23:52
3,1780793350,KDAL,1780803735,KLGA,SWA3256,2026-06-06 20:49,2026-06-06 23:42,2026-06-06,23:42
4,1780796317,KSTL,1780803608,KLGA,SWA4684,2026-06-06 21:38,2026-06-06 23:40,2026-06-06,23:40
...,...,...,...,...,...,...,...,...,...
14577,1783222721,KORD,1783228818,KLGA,AAL2086,2026-07-04 23:38,2026-07-05 01:20,2026-07-05,01:20
14578,1783222641,KCMH,1783227896,KLGA,RPA5784,2026-07-04 23:37,2026-07-05 01:04,2026-07-05,01:04
14579,1783213820,KDFW,1783225456,KLGA,AAL2708,2026-07-04 21:10,2026-07-05 00:24,2026-07-05,00:24
14580,1783223125,KALB,1783224929,KLGA,SWA2144,2026-07-04 23:45,2026-07-05 00:15,2026-07-05,00:15


In [70]:
df_laguardia = df_laguardia[["dep_unix", "dep_conv", "dep_airport", "callsign", "arr_airport", "arrival_unix", "arrival_conv", "arrival", "ar_time"]]

In [73]:
df_laguardia.sort_values("arrival_conv", ascending=False)

,dep_unix,dep_conv,dep_airport,callsign,arr_airport,arrival_unix,arrival_conv,arrival,ar_time
14177,1783296112,2026-07-05 20:01,KMCO,JBU798,KLGA,1783307750,2026-07-05 23:15,2026-07-05,23:15
14178,1783298003,2026-07-05 20:33,KMCO,DAL2520,KLGA,1783307478,2026-07-05 23:11,2026-07-05,23:11
14179,1783296861,2026-07-05 20:14,KMIA,DAL2300,KLGA,1783306631,2026-07-05 22:57,2026-07-05,22:57
14180,1783296921,2026-07-05 20:15,KFLL,DAL2321,KLGA,1783306248,2026-07-05 22:50,2026-07-05,22:50
14181,1783298443,2026-07-05 20:40,KBGR,RPA4348,KLGA,1783305792,2026-07-05 22:43,2026-07-05,22:43
...,...,...,...,...,...,...,...,...,...
317,1780740120,2026-06-06 06:02,CYYZ,EDV5183,KLGA,1780744164,2026-06-06 07:09,2026-06-06,07:09
318,1780742442,2026-06-06 06:40,KALB,EDV5454,KLGA,1780744043,2026-06-06 07:07,2026-06-06,07:07
319,1780740944,2026-06-06 06:15,KBUF,RPA5842,KLGA,1780743914,2026-06-06 07:05,2026-06-06,07:05
320,1780741057,2026-06-06 06:17,KROC,EDV5311,KLGA,1780743687,2026-06-06 07:01,2026-06-06,07:01


In [93]:
df_laguardia.to_csv("df_laguardia.csv", index=False)

In [78]:
arrivals_per_day_laguardia = df_laguardia.groupby("arrival")["arr_airport"].count()
arrivals_per_day_laguardia

arrival
2026-06-06    322
2026-06-07    477
2026-06-08    583
2026-06-09    550
2026-06-10    464
2026-06-11    465
2026-06-12    394
2026-06-13    387
2026-06-14    394
2026-06-15    539
2026-06-16    544
2026-06-17    545
2026-06-18    564
2026-06-19    564
2026-06-20    361
2026-06-21    532
2026-06-22    385
2026-06-23    540
2026-06-24    577
2026-06-25    580
2026-06-26    577
2026-06-27    363
2026-06-28    526
2026-06-29    544
2026-06-30    524
2026-07-01    543
2026-07-02    562
2026-07-03    493
2026-07-04    278
2026-07-05    405
Name: arr_airport, dtype: int64

In [82]:
df_arrivals_per_day_laguardia = pd.DataFrame(arrivals_per_day_laguardia)

In [83]:
df_arrivals_per_day_laguardia

,arr_airport
arrival,
2026-06-06,322
2026-06-07,477
2026-06-08,583
2026-06-09,550
2026-06-10,464
2026-06-11,465
2026-06-12,394
2026-06-13,387
2026-06-14,394


In [81]:
df_laguardia["dep_airport"].value_counts()

dep_airport
KORD    875
CYYZ    620
KBOS    610
KATL    601
KDFW    480
       ... 
KGKY      1
06MO      1
KADW      1
EDSB      1
KNEW      1
Name: count, Length: 255, dtype: int64

## second airport - JFK

In [86]:
airports_ny

{'airport_1': 'John F. Kennedy International Airport (JFK)',
 'ICAO_code_1': 'KJFK',
 'airport_2': 'LaGuardia Airport (LGA)',
 'ICAO_code_2': 'KLGA',
 'airport_3': 'Newark International Airport (EWR)',
 'ICAO_code_3': 'KEWR'}

In [87]:
flights_JFK = get_flights_for_airport("KJFK")

KJFK 1780718400-1780804800: 555 flights found
KJFK 1780804800-1780891200: 670 flights found
KJFK 1780891200-1780977600: 659 flights found
KJFK 1780977600-1781064000: 662 flights found
KJFK 1781064000-1781150400: 607 flights found
KJFK 1781150400-1781236800: 651 flights found
KJFK 1781236800-1781323200: 628 flights found
KJFK 1781323200-1781409600: 661 flights found
KJFK 1781409600-1781496000: 621 flights found
KJFK 1781496000-1781582400: 663 flights found
KJFK 1781582400-1781668800: 662 flights found
KJFK 1781668800-1781755200: 672 flights found
KJFK 1781755200-1781841600: 665 flights found
KJFK 1781841600-1781928000: 685 flights found
KJFK 1781928000-1782014400: 666 flights found
KJFK 1782014400-1782100800: 675 flights found
KJFK 1782100800-1782187200: 595 flights found
KJFK 1782187200-1782273600: 690 flights found
KJFK 1782273600-1782360000: 671 flights found
KJFK 1782360000-1782446400: 692 flights found
KJFK 1782446400-1782532800: 686 flights found
KJFK 1782532800-1782619200: 663 fl

In [89]:
df_jfk= pd.DataFrame(flights_JFK)
df_jfk

,icao24,firstSeen,estDepartureAirport,lastSeen,estArrivalAirport,callsign,estDepartureAirportHorizDistance,estDepartureAirportVertDistance,estArrivalAirportHorizDistance,estArrivalAirportVertDistance,departureAirportCandidatesCount,arrivalAirportCandidatesCount
0,0c6061,1780787746,NaN,1780804781,KJFK,BWA526,NaN,NaN,1279,18,0,1
1,aafc36,1780795969,KFLL,1780804705,KJFK,DAL1419,1526.0,103.0,1224,18,2,1
2,ad2baa,1780776012,NaN,1780804617,KJFK,JBU36,NaN,NaN,1645,34,0,1
3,71c008,1780801713,KBOS,1780804558,KJFK,KAL085,913.0,77.0,1295,18,0,1
4,ab6162,1780800705,KBUF,1780804403,KJFK,EDV5250,714.0,159.0,1235,34,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...
19655,4d0115,1783197347,ELLX,1783227164,KJFK,CLX2,1213.0,25.0,1338,19,0,1
19656,71c252,1783203190,PANC,1783226920,KJFK,KAL257,495.0,15.0,1259,42,7,1
19657,ad27f3,1783208140,TLPL,1783226649,KJFK,JBU1682,14783.0,1687.0,1543,3,0,1
19658,a790b3,1783191037,KJFK,1783226540,KJFK,JBU1112,941.0,34.0,1480,3,0,1


In [94]:
df_jfk = df_jfk.drop(columns = ["icao24", "estDepartureAirportHorizDistance", "estDepartureAirportVertDistance", "estArrivalAirportHorizDistance", "estArrivalAirportVertDistance", "departureAirportCandidatesCount", "arrivalAirportCandidatesCount"])

In [95]:
df_jfk= df_jfk.rename(columns = {"firstSeen" : "dep_unix",
                               "estDepartureAirport" : "dep_airport", 
                               "lastSeen" : "arrival_unix",
                               "estArrivalAirport" : "arr_airport"})
df_jfk

,dep_unix,dep_airport,arrival_unix,arr_airport,callsign
0,1780787746,NaN,1780804781,KJFK,BWA526
1,1780795969,KFLL,1780804705,KJFK,DAL1419
2,1780776012,NaN,1780804617,KJFK,JBU36
3,1780801713,KBOS,1780804558,KJFK,KAL085
4,1780800705,KBUF,1780804403,KJFK,EDV5250
...,...,...,...,...,...
19655,1783197347,ELLX,1783227164,KJFK,CLX2
19656,1783203190,PANC,1783226920,KJFK,KAL257
19657,1783208140,TLPL,1783226649,KJFK,JBU1682
19658,1783191037,KJFK,1783226540,KJFK,JBU1112


In [96]:
df_jfk["arrival_conv"] = (
    pd.to_datetime(df_jfk["arrival_unix"], unit="s", utc=True) #unit = s says that is in seconds
    .dt.tz_convert("America/New_York") #converts to ny timezone                       
    .dt.strftime("%Y-%m-%d %H:%M")
)

In [97]:
df_jfk

,dep_unix,dep_airport,arrival_unix,arr_airport,callsign,arrival_conv
0,1780787746,NaN,1780804781,KJFK,BWA526,2026-06-06 23:59
1,1780795969,KFLL,1780804705,KJFK,DAL1419,2026-06-06 23:58
2,1780776012,NaN,1780804617,KJFK,JBU36,2026-06-06 23:56
3,1780801713,KBOS,1780804558,KJFK,KAL085,2026-06-06 23:55
4,1780800705,KBUF,1780804403,KJFK,EDV5250,2026-06-06 23:53
...,...,...,...,...,...,...
19655,1783197347,ELLX,1783227164,KJFK,CLX2,2026-07-05 00:52
19656,1783203190,PANC,1783226920,KJFK,KAL257,2026-07-05 00:48
19657,1783208140,TLPL,1783226649,KJFK,JBU1682,2026-07-05 00:44
19658,1783191037,KJFK,1783226540,KJFK,JBU1112,2026-07-05 00:42


In [98]:
df_jfk["dep_conv"] = (
    pd.to_datetime(df_jfk["dep_unix"], unit="s", utc=True) #unit = s says that is in seconds
    .dt.tz_convert("America/New_York") #converts to ny timezone                       
    .dt.strftime("%Y-%m-%d %H:%M")
)

In [99]:
df_jfk

,dep_unix,dep_airport,arrival_unix,arr_airport,callsign,arrival_conv,dep_conv
0,1780787746,NaN,1780804781,KJFK,BWA526,2026-06-06 23:59,2026-06-06 19:15
1,1780795969,KFLL,1780804705,KJFK,DAL1419,2026-06-06 23:58,2026-06-06 21:32
2,1780776012,NaN,1780804617,KJFK,JBU36,2026-06-06 23:56,2026-06-06 16:00
3,1780801713,KBOS,1780804558,KJFK,KAL085,2026-06-06 23:55,2026-06-06 23:08
4,1780800705,KBUF,1780804403,KJFK,EDV5250,2026-06-06 23:53,2026-06-06 22:51
...,...,...,...,...,...,...,...
19655,1783197347,ELLX,1783227164,KJFK,CLX2,2026-07-05 00:52,2026-07-04 16:35
19656,1783203190,PANC,1783226920,KJFK,KAL257,2026-07-05 00:48,2026-07-04 18:13
19657,1783208140,TLPL,1783226649,KJFK,JBU1682,2026-07-05 00:44,2026-07-04 19:35
19658,1783191037,KJFK,1783226540,KJFK,JBU1112,2026-07-05 00:42,2026-07-04 14:50


In [101]:
df_jfk[["arrival", "time"]] = df_jfk["arrival_conv"].str.split(" ", expand=True)

In [102]:
df_jfk

,dep_unix,dep_airport,arrival_unix,arr_airport,callsign,arrival_conv,dep_conv,arrival,time
0,1780787746,NaN,1780804781,KJFK,BWA526,2026-06-06 23:59,2026-06-06 19:15,2026-06-06,23:59
1,1780795969,KFLL,1780804705,KJFK,DAL1419,2026-06-06 23:58,2026-06-06 21:32,2026-06-06,23:58
2,1780776012,NaN,1780804617,KJFK,JBU36,2026-06-06 23:56,2026-06-06 16:00,2026-06-06,23:56
3,1780801713,KBOS,1780804558,KJFK,KAL085,2026-06-06 23:55,2026-06-06 23:08,2026-06-06,23:55
4,1780800705,KBUF,1780804403,KJFK,EDV5250,2026-06-06 23:53,2026-06-06 22:51,2026-06-06,23:53
...,...,...,...,...,...,...,...,...,...
19655,1783197347,ELLX,1783227164,KJFK,CLX2,2026-07-05 00:52,2026-07-04 16:35,2026-07-05,00:52
19656,1783203190,PANC,1783226920,KJFK,KAL257,2026-07-05 00:48,2026-07-04 18:13,2026-07-05,00:48
19657,1783208140,TLPL,1783226649,KJFK,JBU1682,2026-07-05 00:44,2026-07-04 19:35,2026-07-05,00:44
19658,1783191037,KJFK,1783226540,KJFK,JBU1112,2026-07-05 00:42,2026-07-04 14:50,2026-07-05,00:42


In [103]:
df_jfk = df_jfk[["dep_unix", "dep_conv", "dep_airport", "callsign", "arr_airport", "arrival_unix", "arrival_conv", "arrival", "time"]]  

In [105]:
df_jfk.to_csv("df_jfk.csv", index=False)

## third airport - newark

In [22]:
flights_newark = get_flights_for_airport("KEWR")

KEWR 1780718400-1780804800: 507 flights found
KEWR 1780804800-1780891200: 631 flights found
KEWR 1780891200-1780977600: 625 flights found
KEWR 1780977600-1781064000: 684 flights found
KEWR 1781064000-1781150400: 581 flights found
KEWR 1781150400-1781236800: 654 flights found
KEWR 1781236800-1781323200: 603 flights found
KEWR 1781323200-1781409600: 651 flights found
KEWR 1781409600-1781496000: 544 flights found
KEWR 1781496000-1781582400: 683 flights found
KEWR 1781582400-1781668800: 681 flights found
KEWR 1781668800-1781755200: 677 flights found
KEWR 1781755200-1781841600: 641 flights found
KEWR 1781841600-1781928000: 687 flights found
KEWR 1781928000-1782014400: 626 flights found
KEWR 1782014400-1782100800: 645 flights found
KEWR 1782100800-1782187200: 554 flights found
KEWR 1782187200-1782273600: 626 flights found
KEWR 1782273600-1782360000: 703 flights found
KEWR 1782360000-1782446400: 693 flights found
KEWR 1782446400-1782532800: 693 flights found
KEWR 1782532800-1782619200: 583 fl

In [23]:
df_newark= pd.DataFrame(flights_newark)
df_newark

,icao24,firstSeen,estDepartureAirport,lastSeen,estArrivalAirport,callsign,estDepartureAirportHorizDistance,estDepartureAirportVertDistance,estArrivalAirportHorizDistance,estArrivalAirportVertDistance,departureAirportCandidatesCount,arrivalAirportCandidatesCount
0,a35ae4,1.780799e+09,KCLT,1780804791,KEWR,AAL2794,6450.0,534.0,1086,47,0,5
1,a1240a,1.780794e+09,KPBI,1780804649,KEWR,UAL1187,3809.0,398.0,1132,55,1,5
2,acb824,1.780788e+09,KSEA,1780804527,KEWR,ASA293,1326.0,12.0,302,9,4,5
3,a1277b,1.780789e+09,NaN,1780804409,KEWR,UAL1446,NaN,NaN,388,9,0,5
4,aaa90c,1.780796e+09,KATL,1780804307,KEWR,RPA3402,876.0,46.0,1050,47,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...
18977,a83782,1.783214e+09,KIAH,1783225777,KEWR,UAL1444,3267.0,244.0,493,28,0,5
18978,0d0acd,1.783205e+09,MMMY,1783225536,KEWR,AMX406,284.0,0.0,972,5,0,5
18979,a2b7f5,1.783213e+09,KIAH,1783224771,KEWR,UAL700,3187.0,259.0,894,13,0,5
18980,ab4f5d,1.783213e+09,KMCO,1783224411,KEWR,JBU1728,2569.0,123.0,367,20,0,5


In [24]:
df_newark = df_newark.drop(columns = ["icao24", "estDepartureAirportHorizDistance", "estDepartureAirportVertDistance", "estArrivalAirportHorizDistance", "estArrivalAirportVertDistance", "departureAirportCandidatesCount", "arrivalAirportCandidatesCount"])

In [25]:
df_newark= df_newark.rename(columns = {"firstSeen" : "dep_unix",
                               "estDepartureAirport" : "dep_airport", 
                               "lastSeen" : "arrival_unix",
                               "estArrivalAirport" : "arr_airport"})
df_newark

,dep_unix,dep_airport,arrival_unix,arr_airport,callsign
0,1.780799e+09,KCLT,1780804791,KEWR,AAL2794
1,1.780794e+09,KPBI,1780804649,KEWR,UAL1187
2,1.780788e+09,KSEA,1780804527,KEWR,ASA293
3,1.780789e+09,NaN,1780804409,KEWR,UAL1446
4,1.780796e+09,KATL,1780804307,KEWR,RPA3402
...,...,...,...,...,...
18977,1.783214e+09,KIAH,1783225777,KEWR,UAL1444
18978,1.783205e+09,MMMY,1783225536,KEWR,AMX406
18979,1.783213e+09,KIAH,1783224771,KEWR,UAL700
18980,1.783213e+09,KMCO,1783224411,KEWR,JBU1728


In [26]:
df_newark["arrival_conv"] = (
    pd.to_datetime(df_newark["arrival_unix"], unit="s", utc=True) #unit = s says that is in seconds
    .dt.tz_convert("America/New_York") #converts to ny timezone                       
    .dt.strftime("%Y-%m-%d %H:%M")
)

In [27]:
df_newark[["arrival", "time"]] = df_newark["arrival_conv"].str.split(" ", expand=True)

In [30]:
df_newark["dep_conv"] = (
    pd.to_datetime(df_newark["dep_unix"], unit="s", utc=True) #unit = s says that is in seconds
    .dt.tz_convert("America/New_York") #converts to ny timezone                       
    .dt.strftime("%Y-%m-%d %H:%M")
)

In [31]:
df_newark.head()

,dep_unix,dep_airport,arrival_unix,arr_airport,callsign,arrival_conv,arrival,time,dep_conv
0,1.780799e+09,KCLT,1780804791,KEWR,AAL2794,2026-06-06 23:59,2026-06-06,23:59,2026-06-06 22:21
1,1.780794e+09,KPBI,1780804649,KEWR,UAL1187,2026-06-06 23:57,2026-06-06,23:57,2026-06-06 21:07
2,1.780788e+09,KSEA,1780804527,KEWR,ASA293,2026-06-06 23:55,2026-06-06,23:55,2026-06-06 19:17
3,1.780789e+09,NaN,1780804409,KEWR,UAL1446,2026-06-06 23:53,2026-06-06,23:53,2026-06-06 19:37
4,1.780796e+09,KATL,1780804307,KEWR,RPA3402,2026-06-06 23:51,2026-06-06,23:51,2026-06-06 21:36


In [32]:
df_newark = df_newark[["dep_unix", "dep_conv", "dep_airport", "callsign", "arr_airport", "arrival_unix", "arrival_conv", "arrival", "time"]]  

In [33]:
df_newark

,dep_unix,dep_conv,dep_airport,callsign,arr_airport,arrival_unix,arrival_conv,arrival,time
0,1.780799e+09,2026-06-06 22:21,KCLT,AAL2794,KEWR,1780804791,2026-06-06 23:59,2026-06-06,23:59
1,1.780794e+09,2026-06-06 21:07,KPBI,UAL1187,KEWR,1780804649,2026-06-06 23:57,2026-06-06,23:57
2,1.780788e+09,2026-06-06 19:17,KSEA,ASA293,KEWR,1780804527,2026-06-06 23:55,2026-06-06,23:55
3,1.780789e+09,2026-06-06 19:37,NaN,UAL1446,KEWR,1780804409,2026-06-06 23:53,2026-06-06,23:53
4,1.780796e+09,2026-06-06 21:36,KATL,RPA3402,KEWR,1780804307,2026-06-06 23:51,2026-06-06,23:51
...,...,...,...,...,...,...,...,...,...
18977,1.783214e+09,2026-07-04 21:16,KIAH,UAL1444,KEWR,1783225777,2026-07-05 00:29,2026-07-05,00:29
18978,1.783205e+09,2026-07-04 18:43,MMMY,AMX406,KEWR,1783225536,2026-07-05 00:25,2026-07-05,00:25
18979,1.783213e+09,2026-07-04 20:58,KIAH,UAL700,KEWR,1783224771,2026-07-05 00:12,2026-07-05,00:12
18980,1.783213e+09,2026-07-04 20:52,KMCO,JBU1728,KEWR,1783224411,2026-07-05 00:06,2026-07-05,00:06


In [34]:
df_newark.to_csv("df_newark.csv", index=False)

## la guardia 2025

In [35]:
start_2025 = datetime(2025, 6, 6)
end_2025 = datetime(2025, 7, 6)

intervals_2025 = []

current_2025 = start_2025

while current_2025 < end_2025:
    next_date_2025 = min(current_2025 + timedelta(days=1), end_2025)

    intervals_2025.append((current_2025, next_date_2025))

    current_2025 = next_date_2025

print(intervals_2025)

[(datetime.datetime(2025, 6, 6, 0, 0), datetime.datetime(2025, 6, 7, 0, 0)), (datetime.datetime(2025, 6, 7, 0, 0), datetime.datetime(2025, 6, 8, 0, 0)), (datetime.datetime(2025, 6, 8, 0, 0), datetime.datetime(2025, 6, 9, 0, 0)), (datetime.datetime(2025, 6, 9, 0, 0), datetime.datetime(2025, 6, 10, 0, 0)), (datetime.datetime(2025, 6, 10, 0, 0), datetime.datetime(2025, 6, 11, 0, 0)), (datetime.datetime(2025, 6, 11, 0, 0), datetime.datetime(2025, 6, 12, 0, 0)), (datetime.datetime(2025, 6, 12, 0, 0), datetime.datetime(2025, 6, 13, 0, 0)), (datetime.datetime(2025, 6, 13, 0, 0), datetime.datetime(2025, 6, 14, 0, 0)), (datetime.datetime(2025, 6, 14, 0, 0), datetime.datetime(2025, 6, 15, 0, 0)), (datetime.datetime(2025, 6, 15, 0, 0), datetime.datetime(2025, 6, 16, 0, 0)), (datetime.datetime(2025, 6, 16, 0, 0), datetime.datetime(2025, 6, 17, 0, 0)), (datetime.datetime(2025, 6, 17, 0, 0), datetime.datetime(2025, 6, 18, 0, 0)), (datetime.datetime(2025, 6, 18, 0, 0), datetime.datetime(2025, 6, 19, 

In [36]:
def to_unix(day):
    return int(day.replace(tzinfo=ZoneInfo("America/New_York")).timestamp())

unix_intervals_2025 = [(to_unix(inicio), to_unix(fim)) for inicio, fim in intervals_2025]

for begin, end in unix_intervals_2025:
    print(begin, end)

1749182400 1749268800
1749268800 1749355200
1749355200 1749441600
1749441600 1749528000
1749528000 1749614400
1749614400 1749700800
1749700800 1749787200
1749787200 1749873600
1749873600 1749960000
1749960000 1750046400
1750046400 1750132800
1750132800 1750219200
1750219200 1750305600
1750305600 1750392000
1750392000 1750478400
1750478400 1750564800
1750564800 1750651200
1750651200 1750737600
1750737600 1750824000
1750824000 1750910400
1750910400 1750996800
1750996800 1751083200
1751083200 1751169600
1751169600 1751256000
1751256000 1751342400
1751342400 1751428800
1751428800 1751515200
1751515200 1751601600
1751601600 1751688000
1751688000 1751774400


In [37]:
len(unix_intervals_2025)

30

In [19]:
CLIENT_ID = client_id_2
CLIENT_SECRET = client_secret_2

TOKEN_URL = "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token"

def get_token():
    r = requests.post(
        TOKEN_URL,
        data={
            "grant_type": "client_credentials",
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
        },
    )
    r.raise_for_status()
    return r.json()["access_token"]

token = get_token()

In [20]:
API_URL = "https://opensky-network.org/api/flights/arrival"
headers = {"Authorization": f"Bearer {token}"}

def get_flights_for_airport(airport):
    
    flights_airport = []
    
    for begin, end in unix_intervals_2025:
        resp = requests.get(
            API_URL,
            params={"airport": airport, "begin": begin, "end": end},
            headers=headers)

        if resp.status_code == 404:
            print(f"No flights were found for {airport} ({begin}-{end})")
            continue

        resp.raise_for_status()
        flights = resp.json()
        print(f"{airport} {begin}-{end}: {len(flights)} flights found")
        flights_airport.extend(flights)
        time.sleep(1)

    return flights_airport  

In [43]:
flights_la_guardia_2025 = get_flights_for_airport("KLGA")

KLGA 1749182400-1749268800: 436 flights found
KLGA 1749268800-1749355200: 342 flights found
KLGA 1749355200-1749441600: 500 flights found
KLGA 1749441600-1749528000: 552 flights found
KLGA 1749528000-1749614400: 531 flights found
KLGA 1749614400-1749700800: 552 flights found
KLGA 1749700800-1749787200: 561 flights found
KLGA 1749787200-1749873600: 572 flights found
KLGA 1749873600-1749960000: 346 flights found
KLGA 1749960000-1750046400: 503 flights found
KLGA 1750046400-1750132800: 553 flights found
KLGA 1750132800-1750219200: 500 flights found
KLGA 1750219200-1750305600: 527 flights found
KLGA 1750305600-1750392000: 406 flights found
KLGA 1750392000-1750478400: 574 flights found
KLGA 1750478400-1750564800: 352 flights found
KLGA 1750564800-1750651200: 493 flights found
KLGA 1750651200-1750737600: 574 flights found
KLGA 1750737600-1750824000: 567 flights found
KLGA 1750824000-1750910400: 498 flights found
KLGA 1750910400-1750996800: 488 flights found
KLGA 1750996800-1751083200: 534 fl

In [45]:
flights_la_guardia_2025[0]

{'icao24': 'a07579',
 'firstSeen': 1749261009,
 'estDepartureAirport': 'KMCO',
 'lastSeen': 1749268709,
 'estArrivalAirport': 'KLGA',
 'callsign': 'DAL2520 ',
 'estDepartureAirportHorizDistance': 9751,
 'estDepartureAirportVertDistance': 610,
 'estArrivalAirportHorizDistance': 434,
 'estArrivalAirportVertDistance': 16,
 'departureAirportCandidatesCount': 0,
 'arrivalAirportCandidatesCount': 3}

In [60]:
df_laguardia_2025 = pd.DataFrame(flights_la_guardia_2025)
df_laguardia_2025

,icao24,firstSeen,estDepartureAirport,lastSeen,estArrivalAirport,callsign,estDepartureAirportHorizDistance,estDepartureAirportVertDistance,estArrivalAirportHorizDistance,estArrivalAirportVertDistance,departureAirportCandidatesCount,arrivalAirportCandidatesCount
0,a07579,1.749261e+09,KMCO,1749268709,KLGA,DAL2520,9751.0,610.0,434,16,0,3
1,aa0759,1.749266e+09,KIAD,1749268625,KLGA,RPA3413,500.0,11.0,601,24,2,3
2,a042db,1.749258e+09,KDFW,1749268499,KLGA,DAL846,1628.0,279.0,407,8,2,3
3,a44cfd,1.749260e+09,KTPA,1749268354,KLGA,DAL2404,1541.0,30.0,385,8,1,3
4,a3a8ef,1.749262e+09,KATL,1749268209,KLGA,FFT3288,1731.0,15.0,362,8,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...
14511,ab0a73,1.751710e+09,KRIC,1751712628,KLGA,RPA5634,2355.0,299.0,423,74,2,3
14512,ab2ae6,1.751708e+09,KPYM,1751711777,KLGA,N819BB,7658.0,366.0,10336,21,1,4
14513,a4417d,1.751677e+09,KDEN,1751690589,KLGA,DAL716,3989.0,66.0,548,74,2,3
14514,ac26cf,1.751684e+09,KMKE,1751690117,KLGA,RPA5850,1099.0,76.0,465,59,0,3


In [66]:
df_laguardia_2025.to_csv("laguardia_2025.csv", index=False)

## JFK 2025

In [54]:
#got blocked, made new credentials

with open("credentials_2.json") as f:
    creds = json.load(f)

CLIENT_ID = creds["client_id"]
CLIENT_SECRET = creds["client_secret"]

TOKEN_URL = "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token"

def get_token():
    r = requests.post(
        TOKEN_URL,
        data={
            "grant_type": "client_credentials",
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
        },
    )
    r.raise_for_status()
    return r.json()["access_token"]

token = get_token()

In [58]:
flights_jfk_2025 = get_flights_for_airport("KJFK")

KJFK 1749182400-1749268800: 644 flights found
KJFK 1749268800-1749355200: 640 flights found
KJFK 1749355200-1749441600: 666 flights found
KJFK 1749441600-1749528000: 658 flights found
KJFK 1749528000-1749614400: 675 flights found
KJFK 1749614400-1749700800: 682 flights found
KJFK 1749700800-1749787200: 703 flights found
KJFK 1749787200-1749873600: 690 flights found
KJFK 1749873600-1749960000: 693 flights found
KJFK 1749960000-1750046400: 668 flights found
KJFK 1750046400-1750132800: 667 flights found
KJFK 1750132800-1750219200: 653 flights found
KJFK 1750219200-1750305600: 637 flights found
KJFK 1750305600-1750392000: 632 flights found
KJFK 1750392000-1750478400: 697 flights found
KJFK 1750478400-1750564800: 697 flights found
KJFK 1750564800-1750651200: 671 flights found
KJFK 1750651200-1750737600: 706 flights found
KJFK 1750737600-1750824000: 707 flights found
KJFK 1750824000-1750910400: 670 flights found
KJFK 1750910400-1750996800: 668 flights found
KJFK 1750996800-1751083200: 680 fl

In [63]:
df_flights_jfk_2025 = pd.DataFrame(flights_jfk_2025)

In [64]:
df_flights_jfk_2025.head(5)

,icao24,firstSeen,estDepartureAirport,lastSeen,estArrivalAirport,callsign,estDepartureAirportHorizDistance,estDepartureAirportVertDistance,estArrivalAirportHorizDistance,estArrivalAirportVertDistance,departureAirportCandidatesCount,arrivalAirportCandidatesCount
0,a85ff5,1.749260e+09,KMCO,1749268639,KJFK,JBU884,8870.0,740.0,1882,18,0,1
1,a66ce0,1.749252e+09,KSEA,1749268421,KJFK,DAL1274,979.0,66.0,1719,18,2,1
2,ab43c6,1.749260e+09,KTPA,1749268314,KJFK,DAL2565,909.0,23.0,1508,3,2,1
3,a6d83c,1.749253e+09,KLAS,1749268237,KJFK,DAL807,1957.0,1.0,1692,18,1,1
4,a657bf,1.749263e+09,KCLT,1749268126,KJFK,AAL1727,2780.0,305.0,1621,11,0,1


In [65]:
df_flights_jfk_2025.to_csv('jfk_2025.csv', index=False)

## newark 2025

In [67]:
newark_2025 = get_flights_for_airport("KEWR")

KEWR 1749182400-1749268800: 544 flights found
KEWR 1749268800-1749355200: 503 flights found
KEWR 1749355200-1749441600: 518 flights found
KEWR 1749441600-1749528000: 467 flights found
KEWR 1749528000-1749614400: 492 flights found
KEWR 1749614400-1749700800: 580 flights found
KEWR 1749700800-1749787200: 592 flights found
KEWR 1749787200-1749873600: 553 flights found
KEWR 1749873600-1749960000: 457 flights found
KEWR 1749960000-1750046400: 501 flights found
KEWR 1750046400-1750132800: 536 flights found
KEWR 1750132800-1750219200: 556 flights found
KEWR 1750219200-1750305600: 560 flights found
KEWR 1750305600-1750392000: 549 flights found
KEWR 1750392000-1750478400: 667 flights found
KEWR 1750478400-1750564800: 536 flights found
KEWR 1750564800-1750651200: 581 flights found
KEWR 1750651200-1750737600: 633 flights found
KEWR 1750737600-1750824000: 596 flights found
KEWR 1750824000-1750910400: 601 flights found
KEWR 1750910400-1750996800: 600 flights found
KEWR 1750996800-1751083200: 605 fl

In [68]:
df_newark_2025 = pd.DataFrame(newark_2025)

In [70]:
df_newark_2025.to_csv('df_newark_2025.csv', index=False)

In [73]:
df_2025 = pd.concat([df_newark_2025, df_laguardia_2025, df_flights_jfk_2025], ignore_index=True)

In [74]:
df_2025.to_csv("df_2025.csv", index=False)